# URLBERT phishing classifier: vonpower + ealvaradob

Second version of the URL model used by the extension. Changes compared to
`train_urlbert_vonpower.ipynb`:

1. **Starts from `CrabInHoney/urlbert-tiny-base-v4`** (pre-trained on URLs
   without phishing labels) instead of the phishing classifier, which had
   already been trained on the whole ealvaradob dataset. Both test sets are
   therefore truly unseen.
2. **Two datasets**: vonpower (2018) and ealvaradob, so the model learns from
   phishing URLs collected from different sources.
3. **No URL scheme**: `http://` and `https://` are removed before training
   and before prediction. In vonpower 97% of phishing URLs use `http://`, so
   the previous model partly learned that `http://` means phishing.
4. **Test URLs are removed from the training data** and both test sets are
   evaluated at the end of the notebook.

## How to run (free Colab is enough)

1. *Runtime > Change runtime type > T4 GPU*.
2. `vonDataset20180426.dill` must be in `MyDrive/phishing_models/` (same as for
   the first notebook). ealvaradob is downloaded automatically.
3. *Runtime > Run all*.

If Colab disconnects, run all cells again: tokenized data is cached on Drive
and training continues from the last checkpoint.

The results are saved to `MyDrive/phishing_models/urlbert_combined_v2/`.
Download that folder and put it into `backend/app/ml/models/`.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "dill"],
        check=True,
    )

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    WORK_DIR = Path("/content/drive/MyDrive/phishing_models")
    VONPOWER_PATH = WORK_DIR / "vonDataset20180426.dill"
    EALVARADOB_DIR = WORK_DIR / "ealvaradob"
else:
    # Local run from backend/ml for testing the notebook.
    BACKEND_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == "ml" else Path.cwd()
    WORK_DIR = BACKEND_DIR / "ml" / "notebook_test_output"
    VONPOWER_PATH = BACKEND_DIR / "app" / "ml" / "data" / "vonDataset20180426.dill"
    EALVARADOB_DIR = BACKEND_DIR / "app" / "ml" / "data" / "ealvaradob"

OUTPUT_DIR = WORK_DIR / "urlbert_combined_v2"
CACHE_DIR = WORK_DIR / "urlbert_combined_v2_cache"
CHECKPOINT_PATH = CACHE_DIR / "checkpoint.pt"

for directory in (OUTPUT_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", VONPOWER_PATH.exists(), VONPOWER_PATH)

In [ ]:
import random

import numpy as np
import torch

BASE_MODEL_NAME = "CrabInHoney/urlbert-tiny-base-v4"

MAX_LENGTH = 64
CHUNK_OVERLAP = 16

EPOCHS = 2
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
WARMUP_SHARE = 0.03
WEIGHT_DECAY = 0.01
CHECKPOINT_EVERY = 2000
SEED = 42

# Small subset for checking that the notebook works end to end.
QUICK_TEST = not IN_COLAB
QUICK_TEST_URLS = 3000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif IN_COLAB:
    print("GPU is not available. Enable Runtime > Change runtime type > T4 GPU.")

## Data

URLs are normalized by removing the scheme. The ealvaradob dataset has no
official split, so it is split by a hash of the URL: 10% test, 10%
validation, 80% training. The same split is used by
`backend/ml/evaluate_urlbert_ealvaradob.py`, so the local evaluation uses
exactly the same test URLs.

In [ ]:
import hashlib
import json

import dill


def normalize_url(url):
    # Must match predictor.preprocess_url in the backend.
    url = url.strip()

    for scheme in ("https://", "http://"):
        if url.lower().startswith(scheme):
            return url[len(scheme):]

    return url


def ealvaradob_split(url):
    # Must match backend/ml/evaluate_urlbert_ealvaradob.py
    bucket = int(hashlib.md5(url.encode("utf-8")).hexdigest(), 16) % 10

    if bucket == 0:
        return "test"

    if bucket == 1:
        return "val"

    return "train"


def decode_vonpower(encoded_urls, char_to_int):
    encoded_urls = np.asarray(encoded_urls)

    lookup = np.full(max(int(v) for v in char_to_int.values()) + 1, "", dtype="<U1")

    for character, value in char_to_int.items():
        lookup[int(value)] = str(character)

    lookup[0] = ""

    rows = lookup[encoded_urls].view(f"<U{encoded_urls.shape[1]}").reshape(-1)

    return [normalize_url(row.replace("\x00", "")) for row in rows]


def remove_conflicts(urls, labels):
    # Keep each URL once, drop URLs that appear with both labels.
    seen = {}

    for url, label in zip(urls, labels):
        if url:
            seen.setdefault(url, set()).add(int(label))

    kept = [(url, next(iter(found))) for url, found in seen.items() if len(found) == 1]

    return [url for url, _ in kept], np.array([label for _, label in kept], dtype=np.int64)


print("Loading vonpower...")

with VONPOWER_PATH.open("rb") as file:
    vonpower = dill.load(file)

splits = {}

for split in ("train", "val", "test"):
    urls = decode_vonpower(vonpower[f"{split}_x"], vonpower["char_to_int"])
    labels = np.asarray(vonpower[f"{split}_y"]).reshape(-1)
    splits[("vonpower", split)] = remove_conflicts(urls, labels)

del vonpower

print("Loading ealvaradob...")

ealvaradob_file = EALVARADOB_DIR / "urls.json"

if not ealvaradob_file.exists():
    from huggingface_hub import hf_hub_download

    hf_hub_download(
        repo_id="ealvaradob/phishing-dataset",
        repo_type="dataset",
        filename="urls.json",
        local_dir=EALVARADOB_DIR,
    )

with ealvaradob_file.open("r", encoding="utf-8") as file:
    records = json.load(file)

ealvaradob_urls, ealvaradob_labels = remove_conflicts(
    [normalize_url(record["text"]) for record in records],
    [record["label"] for record in records],
)

del records

for split in ("train", "val", "test"):
    indices = [i for i, url in enumerate(ealvaradob_urls) if ealvaradob_split(url) == split]
    splits[("ealvaradob", split)] = (
        [ealvaradob_urls[i] for i in indices],
        ealvaradob_labels[indices],
    )

# No URL from a test or validation set may be used for training.
held_out = set()

for (dataset, split), (urls, _) in splits.items():
    if split != "train":
        held_out.update(urls)

train_urls, train_labels = [], []

for dataset in ("vonpower", "ealvaradob"):
    urls, labels = splits[(dataset, "train")]

    for url, label in zip(urls, labels):
        if url not in held_out:
            train_urls.append(url)
            train_labels.append(label)

train_urls, train_labels = remove_conflicts(train_urls, train_labels)

if QUICK_TEST:
    rng = np.random.default_rng(SEED)
    keep = rng.choice(len(train_urls), QUICK_TEST_URLS, replace=False)
    train_urls = [train_urls[i] for i in keep]
    train_labels = train_labels[keep]

    for key, (urls, labels) in splits.items():
        keep = rng.choice(len(urls), min(500, len(urls)), replace=False)
        splits[key] = ([urls[i] for i in keep], labels[keep])

print("Training URLs:", len(train_urls), "labels:", np.bincount(train_labels))

for (dataset, split), (urls, labels) in splits.items():
    if split != "train":
        print(f"{dataset} {split}: {len(urls)} URLs, labels {np.bincount(labels)}")

## Tokenization

All URLs are tokenized once and split into chunks of at most 64 tokens with
an overlap of 16, the same way as in `backend/app/ml/predictor.py`. The result
is cached on Drive.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

CONTENT_LENGTH = MAX_LENGTH - 2
STEP = CONTENT_LENGTH - CHUNK_OVERLAP
PAD_ID = tokenizer.pad_token_id or 0


def split_into_chunks(token_ids):
    if len(token_ids) <= CONTENT_LENGTH:
        return [token_ids]

    chunks = []

    for start in range(0, len(token_ids), STEP):
        chunk = token_ids[start:start + CONTENT_LENGTH]

        if chunk:
            chunks.append(chunk)

        if start + CONTENT_LENGTH >= len(token_ids):
            break

    return chunks


def tokenize_urls(urls, batch_size=100_000):
    """Returns chunk token ids [chunks, 64] and the URL index of every chunk."""

    chunk_parts = []
    owner_parts = []

    for start in range(0, len(urls), batch_size):
        batch = urls[start:start + batch_size]
        token_lists = tokenizer(batch, add_special_tokens=False)["input_ids"]
        chunk_lists = [split_into_chunks(token_ids) for token_ids in token_lists]

        # Written directly into a numpy array, Python lists of this size
        # would need several GB of RAM.
        rows = np.full((sum(map(len, chunk_lists)), MAX_LENGTH), PAD_ID, dtype=np.int16)
        owners = np.empty(len(rows), dtype=np.int64)
        row_index = 0

        for offset, chunks in enumerate(chunk_lists):
            for chunk in chunks:
                rows[row_index, 0] = tokenizer.cls_token_id
                rows[row_index, 1:len(chunk) + 1] = chunk
                rows[row_index, len(chunk) + 1] = tokenizer.sep_token_id
                owners[row_index] = start + offset
                row_index += 1

        chunk_parts.append(rows)
        owner_parts.append(owners)

        print(f"  tokenized {min(start + batch_size, len(urls))}/{len(urls)}")

    return np.concatenate(chunk_parts), np.concatenate(owner_parts)


def cached_tokenize(name, urls):
    path = CACHE_DIR / f"{name}{'_quick' if QUICK_TEST else ''}.npz"

    if path.exists():
        data = np.load(path)

        if data["url_count"] == len(urls):
            return data["chunks"], data["owners"]

    print(f"Tokenizing {name}...")
    chunks, owners = tokenize_urls(urls)
    np.savez(path, chunks=chunks, owners=owners, url_count=len(urls))

    return chunks, owners


train_chunks, train_owners = cached_tokenize("train", train_urls)

eval_data = {
    key: cached_tokenize(f"{key[0]}_{key[1]}", urls)
    for key, (urls, _) in splits.items()
    if key[1] != "train"
}

print("Training chunks:", len(train_chunks))

## Training

Every chunk gets the label of its URL. Long URLs consist of several chunks,
so each chunk is weighted with `1 / number of chunks` to give every URL the
same weight. A checkpoint is saved to Drive every 2000 steps.

In [ ]:
import math
import time

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    id2label={0: "legitimate", 1: "phishing"},
    label2id={"legitimate": 0, "phishing": 1},
)
model.to(device)

# The whole training set fits into GPU memory as int16 (about 300 MB).
chunk_tensor = torch.from_numpy(train_chunks).to(device)
chunk_labels = torch.from_numpy(train_labels[train_owners]).to(device)
chunks_per_url = np.bincount(train_owners, minlength=len(train_urls))
chunk_weights = torch.from_numpy(1.0 / chunks_per_url[train_owners]).float().to(device)

steps_per_epoch = math.ceil(len(train_chunks) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_SHARE), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

global_step = 0

if CHECKPOINT_PATH.exists() and not QUICK_TEST:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    global_step = checkpoint["global_step"]
    print(f"Resumed from step {global_step}/{total_steps}")


def save_checkpoint():
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "global_step": global_step,
        },
        temporary_path,
    )
    temporary_path.replace(CHECKPOINT_PATH)


print(f"{len(train_chunks)} chunks, {steps_per_epoch} steps per epoch, {total_steps} steps")

In [ ]:
def predict_urls(chunks, owners, url_count, batch_size=2048):
    """Phishing probability per URL, mean of the chunk logits like the predictor."""

    model.eval()
    logit_sums = torch.zeros(url_count, 2, device=device)

    with torch.no_grad():
        for start in range(0, len(chunks), batch_size):
            ids = torch.from_numpy(chunks[start:start + batch_size]).to(device).long()

            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(input_ids=ids, attention_mask=(ids != PAD_ID).long()).logits

            batch_owners = torch.from_numpy(owners[start:start + batch_size]).to(device)
            logit_sums.index_add_(0, batch_owners, logits.float())

    counts = torch.from_numpy(np.bincount(owners, minlength=url_count)).to(device).unsqueeze(1)
    probabilities = torch.softmax(logit_sums / counts, dim=1)[:, 1]

    model.train()

    return probabilities.cpu().numpy()


def compute_metrics(labels, probabilities):
    result = {
        "count": int(len(labels)),
        "roc_auc": round(float(roc_auc_score(labels, probabilities)), 4),
    }

    for threshold in (0.5, 0.7):
        predictions = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()

        result[f"{threshold:.2f}"] = {
            "accuracy": round(float(accuracy_score(labels, predictions)), 4),
            "precision": round(float(precision_score(labels, predictions, zero_division=0)), 4),
            "recall": round(float(recall_score(labels, predictions)), 4),
            "f1": round(float(f1_score(labels, predictions)), 4),
            "false_positive_rate": round(float(fp / (fp + tn)), 4),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        }

    return result


def evaluate(split):
    results = {}

    for dataset in ("vonpower", "ealvaradob"):
        urls, labels = splits[(dataset, split)]
        chunks, owners = eval_data[(dataset, split)]
        probabilities = predict_urls(chunks, owners, len(urls))
        results[dataset] = compute_metrics(labels, probabilities)

        values = results[dataset]["0.50"]
        print(
            f"  {dataset} {split}: ROC-AUC {results[dataset]['roc_auc']}, "
            f"precision {values['precision']}, recall {values['recall']}, "
            f"F1 {values['f1']}, FPR {values['false_positive_rate']}"
        )

    return results

In [ ]:
model.train()
start_time = time.time()
loss_sum = 0.0
loss_count = 0
validation_history = []

start_epoch = global_step // steps_per_epoch

for epoch in range(start_epoch, EPOCHS):
    # Fixed order per epoch so that training can resume after a disconnect.
    generator = torch.Generator(device="cpu").manual_seed(SEED + epoch)
    order = torch.randperm(len(train_chunks), generator=generator).to(device)

    for step in range(global_step - epoch * steps_per_epoch, steps_per_epoch):
        batch = order[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
        ids = chunk_tensor[batch].long()

        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(input_ids=ids, attention_mask=(ids != PAD_ID).long()).logits
            losses = torch.nn.functional.cross_entropy(
                logits.float(), chunk_labels[batch], reduction="none"
            )
            weights = chunk_weights[batch]
            loss = (losses * weights).sum() / weights.sum()

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1
        loss_sum += loss.item()
        loss_count += 1

        if global_step % 500 == 0 or global_step == total_steps:
            elapsed = time.time() - start_time
            print(
                f"Epoch {epoch + 1}, step {global_step}/{total_steps}, "
                f"loss {loss_sum / loss_count:.4f}, {elapsed / 60:.1f} min"
            )
            loss_sum = 0.0
            loss_count = 0

        if global_step % CHECKPOINT_EVERY == 0 and not QUICK_TEST:
            save_checkpoint()

    print(f"Validation after epoch {epoch + 1}:")
    validation_history.append(evaluate("val"))

    if not QUICK_TEST:
        save_checkpoint()

training_minutes = round((time.time() - start_time) / 60, 1)
print("Training time in this session:", training_minutes, "minutes")

## Test evaluation and saving

Both test sets were never used for training. The model is saved together with
`preprocessing.json`, which tells the backend to remove the scheme before
prediction.

In [ ]:
print("Test results:")
test_results = evaluate("test")

model.eval()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

(OUTPUT_DIR / "preprocessing.json").write_text(
    json.dumps(
        {
            "strip_scheme": True,
            "max_length": MAX_LENGTH,
            "chunk_overlap": CHUNK_OVERLAP,
            "aggregation": "mean_logits",
        },
        indent=2,
    )
)

(OUTPUT_DIR / "training_results.json").write_text(
    json.dumps(
        {
            "base_model": BASE_MODEL_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "training_urls": len(train_urls),
            "training_chunks": int(len(train_chunks)),
            "quick_test": QUICK_TEST,
            "validation": validation_history,
            "test": test_results,
        },
        indent=2,
    )
)

print("Model saved to:", OUTPUT_DIR)
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))